# Tahap 4 — Case Solution Reuse

Tujuan tahap ini adalah menggunakan putusan lama sebagai dasar prediksi solusi untuk kasus baru.

Notebook ini sudah disesuaikan dengan ketentuan soal Tahap 4:

1. Dari kasus **top-k**, sistem mengambil amar putusan/ringkasan putusan.
2. Solusi disimpan dalam struktur `{case_id: solusi_text}`.
3. Prediksi dilakukan menggunakan:
   - **Majority vote**
   - **Weighted similarity**
4. Fungsi utama dibuat sesuai format soal:
   ```python
   def predict_outcome(query: str) -> str:
       ...
       return predicted_solution
   ```
5. Demo manual dilakukan pada 5 query.
6. Output disimpan ke:
   ```text
   data/results/predictions.csv
   ```
   dengan kolom utama:
   ```text
   query_id, predicted_solution, top_5_case_ids
   ```

## 1. Import Library dan Load Data

In [1]:
from pathlib import Path
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CASES_PATH = PROJECT_ROOT / "data" / "processed" / "cases.csv"
QUERIES_PATH = PROJECT_ROOT / "data" / "eval" / "queries.json"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not CASES_PATH.exists():
    raise FileNotFoundError("data/processed/cases.csv belum ditemukan. Jalankan Tahap 2 terlebih dahulu.")

if not QUERIES_PATH.exists():
    raise FileNotFoundError("data/eval/queries.json belum ditemukan. Jalankan Tahap 3 terlebih dahulu.")

df = pd.read_csv(CASES_PATH).fillna("")

with QUERIES_PATH.open("r", encoding="utf-8") as f:
    queries = json.load(f)

print("Jumlah kasus:", len(df))
print("Jumlah query:", len(queries))
print("Cases path :", CASES_PATH)
print("Queries path:", QUERIES_PATH)

Jumlah kasus: 48
Jumlah query: 10
Cases path : d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\processed\cases.csv
Queries path: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\eval\queries.json


## 2. Validasi Kolom dan Menyiapkan Teks Retrieval

In [2]:
required_cols = [
    "case_id",
    "ringkasan_fakta",
    "argumen_hukum_utama",
    "pasal",
    "pihak",
    "text_full",
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib belum tersedia: {missing_cols}")

optional_cols = []
if "dasar_hukum" in df.columns:
    optional_cols.append("dasar_hukum")

retrieval_cols = ["ringkasan_fakta", "argumen_hukum_utama", "pasal"] + optional_cols + ["pihak"]

df["text_for_retrieval"] = df[retrieval_cols].astype(str).agg(" ".join, axis=1)

print("Kolom retrieval:", retrieval_cols)
display(df[["case_id", "text_for_retrieval"]].head(3))

Kolom retrieval: ['ringkasan_fakta', 'argumen_hukum_utama', 'pasal', 'pihak']


,case_id,text_for_retrieval
0,case_001,gugatanny a memohon kepada pengadilan negeri j...
1,case_002,"posita gugatan penggugat tersebut, tidak diura..."
2,case_003,gugatannya memohon kepada pengadilan negeri to...


## 3. Representasi Vektor TF-IDF

In [3]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=8000,
    ngram_range=(1, 2),
    min_df=1,
)

case_vectors = tfidf_vectorizer.fit_transform(df["text_for_retrieval"])

print("Matriks TF-IDF:", case_vectors.shape)

Matriks TF-IDF: (48, 8000)


## 4. Fungsi Retrieval Top-k

In [4]:
def retrieve(query: str, k: int = 5) -> list:
    """Mengembalikan List[case_id] top-k paling mirip.

    Fungsi ini dibuat sesuai ketentuan Tahap 3/Tahap 4:
    retrieve(query: str, k: int = 5) -> List[case_id]
    """
    query = str(query).lower().strip()
    if not query:
        return []

    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, case_vectors).flatten()

    k = min(k, len(df))
    top_idx = np.argsort(scores)[::-1][:k]

    return df.iloc[top_idx]["case_id"].tolist()


def retrieve_with_scores(query: str, k: int = 5) -> list:
    """Versi detail retrieval untuk weighted similarity dan penyimpanan predictions.csv."""
    query = str(query).lower().strip()
    if not query:
        return []

    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, case_vectors).flatten()

    k = min(k, len(df))
    top_idx = np.argsort(scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[idx]
        results.append({
            "rank": rank,
            "case_id": row["case_id"],
            "no_perkara": row.get("no_perkara", ""),
            "score": round(float(scores[idx]), 6),
            "solution_label": row.get("solution_label", "lainnya"),
            "pasal": row.get("pasal", ""),
            "pihak": row.get("pihak", ""),
        })

    return results

print("Fungsi retrieve() dan retrieve_with_scores() siap digunakan.")

Fungsi retrieve() dan retrieve_with_scores() siap digunakan.


## 5. Ekstraksi Solusi Kasus

Sesuai instruksi, solusi diambil dari amar putusan atau ringkasan putusan.  
Pada dataset perdata waris ini, solusi diambil dari:

1. `argumen_hukum_utama` sebagai sumber utama.
2. `ringkasan_fakta` sebagai fallback apabila `argumen_hukum_utama` kosong.

Hasil disimpan dalam struktur:

```python
case_solutions = {case_id: solusi_text}
```

In [5]:
case_solutions = {}

for _, row in df.iterrows():
    case_id = row["case_id"]

    solusi_text = str(row.get("argumen_hukum_utama", "")).strip()
    if not solusi_text or solusi_text == "TIDAK DITEMUKAN":
        solusi_text = str(row.get("ringkasan_fakta", "")).strip()

    case_solutions[case_id] = solusi_text

print(f"Jumlah solusi tersimpan: {len(case_solutions)}")
print("Contoh struktur {case_id: solusi_text}:")
for case_id, solusi in list(case_solutions.items())[:1]:
    print(case_id, ":", solusi[:500], "...")

Jumlah solusi tersimpan: 48
Contoh struktur {case_id: solusi_text}:
case_001 : 1. menolak permohonan peninjauan kembali dari para pemohon peninjauan kembali 1. yohana s. nugraheni, 2. irma savira firdaus, s.h. tersebut 2. menghukum para pemohon peninjauan kembali untuk membayar biaya perkara dalam pemeriksaan peninjauan kembali sejumlah rp2.500.000,00 (dua juta lima ratus ribu rupiah) demikian ...


## 6. Majority Vote dan Weighted Similarity

In [6]:
case_label_map = dict(
    zip(df["case_id"], df.get("solution_label", pd.Series(["lainnya"] * len(df))))
)


def majority_vote(top_k_case_ids: list) -> str:
    """Memilih solution_label yang paling banyak muncul pada top-k."""
    labels = [case_label_map.get(case_id, "lainnya") for case_id in top_k_case_ids]

    if not labels:
        return "lainnya"

    return Counter(labels).most_common(1)[0][0]


def weighted_similarity_vote(top_k_results: list) -> str:
    """Memilih solution_label berdasarkan total bobot skor similarity tertinggi."""
    label_scores = defaultdict(float)

    for result in top_k_results:
        case_id = result["case_id"]
        label = case_label_map.get(case_id, "lainnya")
        label_scores[label] += float(result.get("score", 0))

    if not label_scores:
        return "lainnya"

    return max(label_scores, key=label_scores.get)

print("Fungsi majority_vote() dan weighted_similarity_vote() siap digunakan.")

Fungsi majority_vote() dan weighted_similarity_vote() siap digunakan.


## 7. Fungsi Utama `predict_outcome(query: str) -> str`

In [7]:
def predict_outcome(query: str) -> str:
    """Memprediksi solusi kasus baru dan mengembalikan teks solusi.

    Fungsi ini sengaja dibuat return string agar sesuai dengan instruksi soal:

    def predict_outcome(query: str) -> str:
        top_k = retrieve(query, k=5)
        solutions = [case_solutions[c] for c in top_k]
        return predicted_solution
    """
    top_k = retrieve(query, k=5)
    solutions = [case_solutions[c] for c in top_k if c in case_solutions]

    top_k_results = retrieve_with_scores(query, k=5)

    # Strategi utama yang dipakai: weighted similarity.
    predicted_label = weighted_similarity_vote(top_k_results)

    selected_case_id = None
    for result in top_k_results:
        case_id = result["case_id"]
        if case_label_map.get(case_id, "lainnya") == predicted_label:
            selected_case_id = case_id
            break

    if selected_case_id is None and top_k:
        selected_case_id = top_k[0]

    predicted_solution = case_solutions.get(
        selected_case_id,
        solutions[0] if solutions else ""
    )

    return predicted_solution


def predict_outcome_detail(query: str, strategy: str = "weighted") -> dict:
    """Fungsi detail untuk demo, evaluasi, dan pembuatan predictions.csv.

    Fungsi ini tidak menggantikan predict_outcome().
    predict_outcome() tetap return string sesuai soal.
    """
    top_k = retrieve(query, k=5)
    top_k_results = retrieve_with_scores(query, k=5)
    solutions = [case_solutions[c] for c in top_k if c in case_solutions]

    if strategy == "majority":
        predicted_label = majority_vote(top_k)
    elif strategy == "weighted":
        predicted_label = weighted_similarity_vote(top_k_results)
    else:
        raise ValueError("strategy harus 'majority' atau 'weighted'")

    selected_case_id = None
    for result in top_k_results:
        case_id = result["case_id"]
        if case_label_map.get(case_id, "lainnya") == predicted_label:
            selected_case_id = case_id
            break

    if selected_case_id is None and top_k:
        selected_case_id = top_k[0]

    predicted_solution = case_solutions.get(
        selected_case_id,
        solutions[0] if solutions else ""
    )

    return {
        "predicted_solution_label": predicted_label,
        "predicted_solution": predicted_solution,
        "selected_case_id": selected_case_id,
        "top_5_case_ids": top_k,
        "top_5_scores": [r["score"] for r in top_k_results],
        "solutions": solutions,
    }

print("Fungsi predict_outcome() sudah sesuai soal dan return string.")
print("Fungsi predict_outcome_detail() tersedia untuk output detail.")

Fungsi predict_outcome() sudah sesuai soal dan return string.
Fungsi predict_outcome_detail() tersedia untuk output detail.


## 8. Uji Fungsi `predict_outcome()`

In [8]:
if len(queries) > 0:
    sample_query = queries[0]["query_text"]
    sample_prediction = predict_outcome(sample_query)

    print("Tipe output predict_outcome():", type(sample_prediction))
    print("Preview predicted_solution:")
    print(sample_prediction[:1000])
else:
    print("queries.json kosong.")

Tipe output predict_outcome(): <class 'str'>
Preview predicted_solution:
1. menolak permohonan kasasi dari pemohon kasasi alberthina surita tersebut 2. menghukum pemohon kasasi untuk membayar biaya perkara dalam tingkat kasasi sejumlah rp500.000,00 (lima ratus ribu rupiah)


## 9. Demo Manual 5 Query

In [9]:
demo_queries = queries[:5]

demo_rows = []

for q in demo_queries:
    query_id = q.get("query_id", "")
    query_text = q.get("query_text", "")
    ground_truth = q.get("ground_truth_case_id", [""])[0]

    pred = predict_outcome_detail(query_text, strategy="weighted")

    demo_rows.append({
        "query_id": query_id,
        "ground_truth_case_id": ground_truth,
        "predicted_solution_label": pred["predicted_solution_label"],
        "selected_case_id": pred["selected_case_id"],
        "top_5_case_ids": ", ".join(pred["top_5_case_ids"]),
        "top_5_scores": ", ".join([str(s) for s in pred["top_5_scores"]]),
    })

demo_df = pd.DataFrame(demo_rows)
display(demo_df)

,query_id,ground_truth_case_id,predicted_solution_label,selected_case_id,top_5_case_ids,top_5_scores
0,Q_001,case_030,menolak,case_030,"case_030, case_029, case_008, case_038, case_019","0.60597, 0.132745, 0.097153, 0.095663, 0.09348"
1,Q_002,case_043,menolak,case_045,"case_043, case_045, case_038, case_036, case_035","0.46468, 0.139822, 0.137411, 0.137398, 0.137398"
2,Q_003,case_029,menolak,case_029,"case_029, case_038, case_035, case_036, case_030","0.628249, 0.145293, 0.125122, 0.125122, 0.12367"
3,Q_004,case_046,menolak,case_046,"case_046, case_045, case_035, case_036, case_038","0.4642, 0.129888, 0.127485, 0.127485, 0.125651"
4,Q_005,case_027,menolak,case_027,"case_027, case_038, case_045, case_013, case_022","0.421648, 0.197797, 0.17325, 0.158706, 0.155172"


## 10. Simpan Output `predictions.csv`

In [10]:
prediction_rows = []

for q in demo_queries:
    query_id = q.get("query_id", "")
    query_text = q.get("query_text", "")
    ground_truth = q.get("ground_truth_case_id", [""])[0]

    pred = predict_outcome_detail(query_text, strategy="weighted")

    prediction_rows.append({
        "query_id": query_id,
        "ground_truth_case_id": ground_truth,
        "predicted_solution_label": pred["predicted_solution_label"],
        "predicted_solution": pred["predicted_solution"],
        "selected_case_id": pred["selected_case_id"],
        "top_5_case_ids": ", ".join(pred["top_5_case_ids"]),
        "top_5_scores": ", ".join([str(s) for s in pred["top_5_scores"]]),
    })

predictions_df = pd.DataFrame(prediction_rows)

PREDICTIONS_PATH = RESULTS_DIR / "predictions.csv"
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print("predictions.csv berhasil disimpan:", PREDICTIONS_PATH)
display(predictions_df.head())

predictions.csv berhasil disimpan: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\results\predictions.csv


,query_id,ground_truth_case_id,predicted_solution_label,predicted_solution,selected_case_id,top_5_case_ids,top_5_scores
0,Q_001,case_030,menolak,1. menolak permohonan kasasi dari pemohon kasa...,case_030,"case_030, case_029, case_008, case_038, case_019","0.60597, 0.132745, 0.097153, 0.095663, 0.09348"
1,Q_002,case_043,menolak,1. menolak permohonan kasasi dari pemohon kasa...,case_045,"case_043, case_045, case_038, case_036, case_035","0.46468, 0.139822, 0.137411, 0.137398, 0.137398"
2,Q_003,case_029,menolak,1. menolak permohonan kasasi dari pemohon kasa...,case_029,"case_029, case_038, case_035, case_036, case_030","0.628249, 0.145293, 0.125122, 0.125122, 0.12367"
3,Q_004,case_046,menolak,1. menolak permohonan kasasi dari pemohon kasa...,case_046,"case_046, case_045, case_035, case_036, case_038","0.4642, 0.129888, 0.127485, 0.127485, 0.125651"
4,Q_005,case_027,menolak,1. menolak permohonan kasasi dari para pemohon...,case_027,"case_027, case_038, case_045, case_013, case_022","0.421648, 0.197797, 0.17325, 0.158706, 0.155172"


## 11. Kesimpulan Tahap 4

Tahap 4 sudah memenuhi ketentuan soal karena menghasilkan:

1. Struktur `case_solutions = {case_id: solusi_text}`.
2. Fungsi `majority_vote()`.
3. Fungsi `weighted_similarity_vote()`.
4. Fungsi utama `predict_outcome(query: str) -> str` yang **return string**.
5. Demo manual 5 query.
6. File `data/results/predictions.csv` dengan kolom utama:
   - `query_id`
   - `predicted_solution`
   - `top_5_case_ids`

Kolom tambahan seperti `ground_truth_case_id`, `predicted_solution_label`, `selected_case_id`, dan `top_5_scores` disimpan untuk membantu evaluasi Tahap 5.